# AquaPINN Colab — Synthetic Benchmark

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cwq2016/AquaPINN_tmp/blob/main/Examples/Colab/02_Synthetic_Benchmark.ipynb)


Generate one selected synthetic trajectory scenario, simulate acoustic detections, run AquaPINN, and compare the positions with the known synthetic reference.

The run folder uses the Full Study desktop record format and a processing/Track/track result path. Synthetic results are controlled examples, not field-accuracy evidence.

## 1. Source, runtime, and persistent output

Set REPO_URL after the public repository is created. Colab writes to Google Drive by default; local Jupyter writes alongside the notebook.

In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, shutil, subprocess, sys, uuid

REPO_URL = "https://github.com/cwq2016/AquaPINN_tmp.git"
REPO_REF = "main"

def find_repo():
    for p in (Path.cwd(), *Path.cwd().parents):
        if (p / "AquaPINN_Tracker3D").is_dir() and (p / "Examples").is_dir():
            return p
    return None

REPO_DIR = find_repo()
if REPO_DIR is None:
    if not REPO_URL:
        raise RuntimeError("Set REPO_URL to the public repository URL, then rerun this cell.")
    REPO_DIR = Path("/content/aquapinn-source")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF,
                        REPO_URL, str(REPO_DIR)], check=True)
    if not (REPO_DIR / "AquaPINN_Tracker3D").is_dir():
        raise RuntimeError("Repository is missing AquaPINN_Tracker3D/")

try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount("/content/drive")
    OUTPUT_PARENT = Path("/content/drive/MyDrive/AquaPINN_Results")
except ImportError:
    IN_COLAB = False
    OUTPUT_PARENT = Path.cwd() / "AquaPINN_Colab_Runs"
OUTPUT_PARENT.mkdir(parents=True, exist_ok=True)

def new_run(label):
    run_id = uuid.uuid4().hex
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    path = OUTPUT_PARENT / f"{label}_{stamp}_{run_id[:8]}"
    path.mkdir(parents=True, exist_ok=False)
    return path, run_id

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def write_json(path, obj):
    Path(path).write_text(json.dumps(obj, indent=2, default=str) + "\n", encoding="utf-8")

sys.path.insert(0, str(REPO_DIR))

print("Project:", REPO_DIR)
print("Persistent output parent:", OUTPUT_PARENT)


## 2. Install dependencies

In [ ]:
%pip -q install "numpy<2.5" scipy pandas matplotlib utm networkx scikit-learn tqdm six

In [ ]:
import torch, tomllib
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if sys.version_info < (3, 11):
    raise RuntimeError("Python 3.11 or later is recommended.")

In [ ]:
import pandas as pd

def track_digest(path):
    path = Path(path)
    frame = pd.read_csv(path)
    return {
        "csv_sha256": sha256(path),
        "rows": int(len(frame)),
        "columns": [str(c) for c in frame.columns],
        "first_datetime": str(frame["datetime"].dropna().iloc[0]) if "datetime" in frame and frame["datetime"].notna().any() else None,
        "last_datetime": str(frame["datetime"].dropna().iloc[-1]) if "datetime" in frame and frame["datetime"].notna().any() else None,
    }

def make_study(name, method, samples, parameters, tag_file, selected_rows,
               coordinate_frame, source_config):
    table = pd.read_csv(tag_file)
    selected = {int(i) for i in selected_rows}
    tags = []
    for row, record in table.iterrows():
        code = str(record.get("Code", f"Tag{row}")).strip()
        try:
            pri = float(record.get("PRI", 0))
        except (TypeError, ValueError):
            pri = 0.0
        tags.append({"code": code, "row": int(row), "nominal_pri": pri,
                     "role": "fish" if int(row) in selected else "unassigned",
                     "selected": int(row) in selected, "gps": None})
    return {
        "format": "aquapinn-study", "version": 1, "name": name,
        "workflow": "fish", "method": method,
        "nnplus_samples": max(2, int(samples)),
        "parameters": {k: (v if isinstance(v, list) else [v]) for k, v in parameters.items()},
        "tags": tags, "unknown_parameters": {}, "source_config": str(source_config),
        "import_notes": ["Created by an AquaPINN Colab workflow."],
        "coordinate_frame": coordinate_frame, "sync_support_reviewed": True,
        "gps_draft": {"file": "", "utc_offset": 0, "matching_frame": False,
                       "confirmed": False, "start": "", "end": "", "target_codes": []},
        "evaluation": {"window_seconds": 90, "step_seconds": 1,
                       "max_gap_seconds": 5, "constant_pri_confirmed": False},
        "example": False,
    }


## 3. Study setup and main parameters

Select the scenario and method. Generation uses the repository simulator and fixed random seed; the editable seed controls AquaPINN initialization.

In [ ]:
import pandas as pd

RUN_DIR, RUN_ID = new_run("Synthetic")
SETUP = {
    "scenario": "baseline",       # baseline, sudden_turn, sudden_burst, tmp_Exit
    "method": "AML",              # AML, NN, or NNplus
    "nnplus_samples": 5,
    "temperature_c": 8.0,
    "random_seed": 42,
    "device": "Automatic",       # Automatic or CPU only
}
display(pd.DataFrame(SETUP.items(), columns=["Parameter", "Value"]))
print("Run folder:", RUN_DIR)

## 4. Generate and stage one scenario

In [ ]:
import csv
import importlib
import pickle
import shutil
import tomllib
from pathlib import Path

SYN_ROOT = REPO_DIR / "Examples/Synthetic"
GENERATOR = SYN_ROOT / "dataGenerator/caseDesign_CabotSynthetic.py"
with (RUN_DIR / "generator.log").open("w", encoding="utf-8") as log:
    generated = subprocess.run([sys.executable, str(GENERATOR)], cwd=GENERATOR.parent,
                               stdout=log, stderr=subprocess.STDOUT, text=True)
if generated.returncode != 0:
    raise RuntimeError(f"Generator failed; inspect {RUN_DIR / 'generator.log'}")
source_dir = SYN_ROOT / "data/Critical"
scenario = SETUP["scenario"]
decode_file = source_dir / f"{scenario}.pickle"
gps_file = source_dir / f"{scenario}_GPS.csv"
if not decode_file.is_file() or not gps_file.is_file():
    raise FileNotFoundError(f"Generated scenario data are missing for {scenario}")

INPUTS = RUN_DIR / "inputs"
PROCESSING = RUN_DIR / "processing"
INPUTS.mkdir()
PROCESSING.mkdir()
decode_copy = INPUTS / decode_file.name
gps_copy = INPUTS / gps_file.name
shutil.copy2(decode_file, decode_copy)
shutil.copy2(gps_file, gps_copy)

phone_file = SYN_ROOT / "dataGenerator/Cabot_Station_hydrophone_configuration_update_3.csv"
tag_file = PROCESSING / "synthetic_tags.csv"
with tag_file.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.writer(stream)
    writer.writerow(["Code", "PRI"])
    writer.writerow(["SYNTHETIC_TAG", 2.0])
shutil.copy2(phone_file, PROCESSING / f"{phone_file.stem}_synced.csv")
shutil.copy2(decode_copy, PROCESSING / "synthetic_tags_TagDecodes_synced.pickle")

with (SYN_ROOT / "run/params.toml").open("rb") as stream:
    params = tomllib.load(stream)
method = SETUP["method"]
engine_method = "AML" if method == "AML" else "NN"
samples = 1 if method != "NNplus" else int(SETUP["nnplus_samples"])
params.update({
    "output_folder": [str(PROCESSING)],
    "phoneFile": [str(phone_file)],
    "tagFile": [str(tag_file)],
    "GPSFile": [str(gps_copy)],
    "dataFolder": [str(INPUTS)],
    "TagID": [0],
    "trackMethod": [engine_method],
    "nSamples": [samples],
    "temperature_ref": [float(SETUP["temperature_c"])],
})
params_file = RUN_DIR / "params_effective.json"
write_json(params_file, params)
write_json(RUN_DIR / "source-input-hashes.json", {
    str(decode_copy): sha256(decode_copy),
    str(gps_copy): sha256(gps_copy),
    str(phone_file): sha256(phone_file),
})
TRACK_REL = f"processing/Track/track/track_Tag0_SYNTHETIC_TAG_{method}.csv"
print("Scenario:", scenario, "| Method:", method, "| Fits:", samples)
print("Staged inputs:", decode_copy, gps_copy)

## 5. Track and save a desktop-style run

In [ ]:
import importlib
from copy import deepcopy

package = importlib.import_module("AquaPINN_Tracker3D")
config = importlib.import_module("AquaPINN_Tracker3D.config")
config.oneKey_configure(use_cuda=(SETUP["device"] == "Automatic"), precision=64,
                        seed=int(SETUP["random_seed"]))
package.getLogger(str(RUN_DIR / "engine.log"))
loc = package.localizationSystem().deploy(str(params_file), doSync=False)
try:
    package.tracker.run(str(params_file), loc)
except Exception as error:
    write_json(RUN_DIR / "worker.json", {"run_id": RUN_ID, "state": "failed", "error": str(error)})
    raise

track_csv = RUN_DIR / TRACK_REL
if not track_csv.is_file() or not Path(str(track_csv) + ".pickle").is_file():
    raise FileNotFoundError("Expected track CSV and pickle sidecar were not created.")

study_tags = [{
    "code": "SYNTHETIC_TAG", "row": 0, "nominal_pri": 2.0,
    "role": "fish", "selected": True, "gps": None,
}]
study = {
    "format": "aquapinn-study", "version": 1, "name": f"Synthetic {scenario}",
    "workflow": "fish", "method": method, "nnplus_samples": max(2, samples),
    "parameters": params, "tags": study_tags, "unknown_parameters": {},
    "source_config": str(params_file), "import_notes": ["Created by the AquaPINN Colab synthetic workflow."],
    "coordinate_frame": "Synthetic local Cartesian coordinates in metres.",
    "sync_support_reviewed": True,
    "gps_draft": {"file": str(gps_copy), "utc_offset": 0, "matching_frame": False,
                  "confirmed": False, "start": "", "end": "", "target_codes": []},
    "evaluation": {"window_seconds": 90, "step_seconds": 1, "max_gap_seconds": 5,
                   "constant_pri_confirmed": False},
    "example": False,
}
summary = track_digest(track_csv)
record = {
    "format": "aquapinn-study-run", "version": 1, "run_id": RUN_ID,
    "state": "completed", "method": method, "study": study, "scenario": scenario,
    "jobs": [{
        **deepcopy(study_tags[0]), "method": method, "state": "completed",
        "track_csv": TRACK_REL, "track_sha256": summary["csv_sha256"], "summary": summary,
    }],
}
write_json(RUN_DIR / "study.json", study)
write_json(RUN_DIR / "run.json", record)
write_json(RUN_DIR / "worker.json", {
    "run_id": RUN_ID, "state": "completed", "stage": "Completed",
    "scenario": scenario, "method": method, "track_csv": TRACK_REL,
    "track_sha256": summary["csv_sha256"],
})
print("Completed:", track_csv)
print("Run record:", RUN_DIR / "run.json")

## 6. Compare with synthetic truth

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display

gps = pd.read_csv(gps_copy)
track = pd.read_csv(track_csv)
# Both files are in the global Easting/Northing frame: the tracker converts
# its local solution back to ENE before writing the track CSV.
valid_track = track[["X", "Y"]].notna().all(axis=1)
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(gps["Easting"], gps["Northing"], color="red", linewidth=1, label="Synthetic truth")
ax.scatter(track.loc[valid_track, "X"], track.loc[valid_track, "Y"],
           s=8, label="AquaPINN positions")
ax.set(xlabel="Easting (m)", ylabel="Northing (m)", title=f"{scenario} — {method}")
ax.set_aspect("equal", adjustable="datalim")
ax.grid(alpha=.25)
ax.legend()
plt.show()
print(f"Tracked positions: {int(valid_track.sum())} of {len(track)} rows")
display(track.head())
print("Complete run folder:", RUN_DIR)
